# 08. 세 조건 평가와 실패 해석 / 60분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

실행 완료와 과제 성공은 다른 열입니다. 결과가 없으면 미실행으로 남기고 성공률 분모에 섞지 않습니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_lab08_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![세 조건 평가와 실패 해석의 입력·처리·출력](../교재그림/guided-vla/day5-08.svg)

각 실행 폴더의 결과를 읽고 rollout만 집계합니다. completed와 success를 분리하고 오류·중단·파일 없음도 표시합니다. A↔B는 horizon, A↔C는 초기 배치만 다릅니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![5일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day5-system.svg)

브라우저는 실행을 요청하고 결과를 표시합니다. 커널이 호출한 별도 runner가 물리 환경과 정책을 연결합니다. 01·02의 합성 셀은 구조 설명용이고 실제 환경 실행은 04, 모델 실행은 06부터입니다. 아래 단원별 도식에서 이번에 실제로 실행할 범위를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 완료·성공·미실행

- completed는 실행 절차가 끝났다는 상태입니다.
- success는 환경이 판정한 과제 달성 여부입니다.
- incomplete는 오류·중단이며 완료 실패와 다릅니다.
- 파일이 없으면 미실행으로 남기고 분모를 구분합니다.

실행이 스텝 한도까지 정상 진행했지만 목표를 달성하지 못하면 completed=True에 해당하는 status와 success=False를 가집니다. 코드 오류나 창을 조기에 닫아 중단되면 incomplete입니다. 결과 파일 자체가 없다면 미실행입니다. 이 세 경우를 모두 실패 한 종류로 묶으면 환경 준비 문제와 정책의 과제 수행을 구분할 수 없습니다.

inspect와 infer도 정상 완료 상태를 가질 수 있지만 과제 수행 rollout이 아니므로 성공률 집계에 넣지 않습니다. summarize는 전달된 결과의 상태를 집계하므로 호출하는 쪽에서 평가할 mode와 조건을 올바르게 고르는 책임이 있습니다. 함수 이름만 보고 모든 입력을 알아서 정제한다고 가정하지 않습니다.

성공률은 분자와 분모를 명시해야 합니다. 예를 들어 완료 rollout 두 건 중 한 건 성공이면 완료 기준 1/2입니다. 여기에 미실행 한 건이 있다고 같은 의미의 1/3이라고 쓰지 않습니다. 요청 조건 수, 완료 수, 중단 수, 미실행 수를 함께 보여 주면 누락이 숨겨지지 않습니다.


### 2. 실행 아키텍처: 생성 셀과 읽기 셀 분리

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day5-08.svg)

- A=초기0/h10, B=초기0/h5, C=초기1/h10
- 비교 실행 셀: 결과 파일 없는 조건만 실행
- 평가 셀: 기존 JSON·영상만 읽고 미실행 표시
- summarize → 자동 표; 기존 중단 결과 보존

08에는 실제 실행을 생성하는 셀과 결과를 읽는 셀이 구분되어 있습니다. 첫 셀은 필요한 조건의 result.json이 없을 때만 순서대로 run_lesson을 호출합니다. 이미 incomplete가 있으면 자동으로 지우고 다시 돌리지 않습니다. 재시도가 필요하면 새로운 실행 이름으로 기록합니다.

평가 셀은 run_missing=False와 allow_incomplete=True로 기존 자료를 읽습니다. 따라서 표를 다시 표시하는 동작이 무거운 정책 실행을 몰래 시작하지 않습니다. 다만 실행 조건 일치 검사는 그대로 유지되므로 파일이 있다고 다른 seed나 horizon 결과를 섞어 읽지 않습니다.

각 행은 조건과 결과 파일·영상 링크를 대응합니다. 같은 SESSION을 사용해야 앞선 04~07의 결과가 이어집니다. SESSION이 달라 비어 있다면 정책 실패가 아니라 다른 실험 폴더를 보고 있는 상황일 수 있습니다.


### 3. 코드 추적: 공정한 두 비교

- A↔B는 execute만 변경하여 horizon 영향을 봅니다.
- A↔C는 초기 배치만 변경합니다.
- B↔C는 두 조건이 달라 원인을 단정할 수 없습니다.
- 중단·재시도 이름과 조건을 보존해 결과를 추적합니다.

A와 B는 초기0·seed7·모델·최대280스텝을 고정하고 execute만 10과 5로 바꿉니다. A와 C는 execute10을 고정하고 초기 배치만 0과 1로 바꿉니다. B와 C를 직접 비교하면 horizon과 초기 배치가 동시에 달라져 어느 변화가 결과를 만들었는지 분리하기 어렵습니다.

run_or_read_lesson은 기존 결과의 mode, seed, task_id, initial_state_index, execution_horizon, requested_steps를 요청값과 대조합니다. 이름이 seed7-h10이라고 써 있어도 실제 JSON의 조건이 다르면 재사용하지 않습니다. 사람이 정한 폴더 이름과 실행 기록을 교차 확인하는 장치입니다.

summarize의 completed 목록에 대해서만 successes와 success_rate_completed를 계산합니다. 완료가 0건이면 비율은 None입니다. 아직 측정하지 않은 비율을 0%로 채우면 모두 실패했다는 잘못된 의미가 됩니다. 자동 표의 빈 상태도 정보로 읽습니다.


### 4. 실패 해석과 4·5일차 연결

- 환경·입력 변환·추론·행동·평가 경계를 순서대로 확인
- 영상에서 실패 시점과 action·종료 이유를 대응
- 소수 조건의 결과를 전체 성공률로 일반화하지 않음
- 4일차의 명시적 제어와 5일차의 학습 행동을 비교

실패를 설명할 때 모델이 나쁘다는 한 문장으로 끝내지 않습니다. 초기 장면이 맞는지, 영상 방향과 상태 계약이 맞는지, 예측이 유한한지, 실제 행동이 적용됐는지, 성공 정보가 정상인지 경계별 근거를 확인합니다. 물체에 접근하지 못한 경우와 접근했으나 잡지 못한 경우는 같은 success=False라도 관찰되는 단계가 다릅니다.

현재 세 조건은 구조를 이해하는 소수 비교입니다. 일반 성능을 추정하려면 더 많은 초기상태와 반복, 고정된 정책·조건, 사전에 정한 평가 기준이 필요합니다. 성공할 때까지 반복한 마지막 결과만 남기면 비교가 편향되므로 원래 중단·실패와 재시도를 함께 보존합니다.

4일차는 명시적인 판정 계약과 사람이 작성한 목표 제어가 중심이고, 5일차는 학습된 수치 행동 예측과 환경 피드백이 중심입니다. 두 경우 모두 관측 표현, 시간·실행 경계, 출력 계약, 실패 근거가 중요합니다. 수강생이 답안을 작성하지 않아도 강사가 자동 표와 영상을 이 순서로 해설하면 전체 실행 흐름을 다시 연결할 수 있습니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [scripts/course_runtime.py : summarize](scripts/course_runtime.py) |
| 바꾸어 볼 조건 | `초기1/h10 추가; task·seed·steps 유지` |
| 예상되는 차이 | 완료 조건 수·미실행 이름·작업 성공을 자동 출력합니다. 각 조건 한 번의 결과는 탐색 비교이며 전체 성공률 추정에는 반복이 더 필요합니다. |
| 직접 구현·수정할 때 | 중단 결과를 덮어쓰지 않습니다. 재시도는 새 이름으로 보존하고 평가 목록에도 추가합니다. 같은 조건 완료 결과는 재사용하며 성공할 때까지 반복하지 않습니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


### 실제 코드 읽기 — 입력·처리·출력 연결

각 실행 폴더의 결과를 읽고 rollout만 집계합니다. completed와 success를 분리하고 오류·중단·파일 없음도 표시합니다. A↔B는 horizon, A↔C는 초기 배치만 다릅니다.


In [ ]:
from pathlib import Path
from IPython.display import Code, display
import ast
source_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'10.VLA_v2/scripts').is_dir())
source_path = source_root/'10.VLA_v2'/'scripts/course_runtime.py'
text = source_path.read_text()
# 함수 위치를 AST로 찾아 해당 함수 전체를 표시합니다. 파일을 import하거나 모델을 로드하지 않습니다.
functions = [node for node in ast.parse(text).body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
for node in functions:
    if node.name in ['summarize', 'run_or_read_lesson']:
        print(source_path, '함수:', node.name, '줄:', node.lineno)
        display(Code('\n'.join(text.splitlines()[node.lineno-1:node.end_lineno]), language='python'))


## 준비 — 앞 단원과 같은 SESSION

초기0/h10, 초기0/h5, 초기1/h10을 비교합니다. 같은 조건의 완료·중단 결과는 보존합니다. 아래 셀은 없는 조건만 실행하므로 첫 두 조건을 아직 실행하지 않았다면 그만큼 시간이 더 필요합니다.


In [ ]:
from pathlib import Path
import sys, subprocess, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA_v2').is_dir())
SCRIPTS = ROOT / '10.VLA_v2/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson, summarize
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
GUI = os.environ.get('COURSE_VLA_GUI', '0') == '1'  # 개인 noVNC 화면 사용
SESSION = 'v2-session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
EXPERIMENTS = [
    ("seed7-h10", dict(execute=10)),
    ("seed7-h5", dict(execute=5)),
    ("init1-h10", dict(initial_state=1)),
]


## 비교 실행 — 누락 조건만 차례로 실행

한 번에 한 정책 프로세스만 실행합니다. 각 FINISHED 화면에서 MuJoCo 창을 닫아야 다음 조건으로 넘어갑니다. 중단 결과가 이미 있으면 덮어쓰지 않고 평가표에 표시합니다. 재시도는 강사 안내에 따라 새 실행 이름과 같은 조건을 EXPERIMENTS에 추가합니다.


In [ ]:
# 실제 실험을 실행하는 셀입니다. 읽기만 원할 때는 이 셀을 건너뛰고 아래 평가 셀을 실행합니다.
# 중단 결과가 있는 폴더도 보존하고 없는 결과만 생성합니다.
for name, conditions in EXPERIMENTS:
    if not (RUNS/name/'result.json').exists():
        print('새 비교 실행:', name, conditions)
        try:
            run_or_read_lesson(SCRIPTS, RUNS, 'rollout', name, gui=GUI, **conditions)
        except subprocess.CalledProcessError as exc:
            print('실험 실패:', name, exc, '— 남은 조건은 평가표에서 미실행으로 확인합니다.')
            break
    else:
        print('보존할 기존 결과:', name)


## 결과표 자동 생성 — 완료와 성공은 별도 열


In [ ]:
# 기존 완료·중단 기록만 읽습니다. 없는 실험은 실행하지 않고 별도로 표시합니다.
rows, result_names, missing_names = [], [], []
for name, conditions in EXPERIMENTS:
    row = run_or_read_lesson(SCRIPTS, RUNS, 'rollout', name,
                            allow_incomplete=True, run_missing=False, **conditions)
    if row is None:
        missing_names.append(name)
    else:
        rows.append(row)
        result_names.append(name)
summary = {'summary': summarize(rows), 'runs': rows,
           'run_names': result_names, 'missing_runs': missing_names}
(RUNS/'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
print('아직 result.json이 없는 실험:', missing_names)
summary['summary']


In [ ]:
from IPython.display import Markdown, Video, display
lines = ['실행 | 초기 | horizon | 실행 상태 | 작업 성공 | 스텝', '--- | --- | --- | --- | --- | ---']
by_name = dict(zip(summary['run_names'], summary['runs']))
for name, conditions in EXPERIMENTS:
    row = by_name.get(name)
    if row is None:
        lines.append(f'{name} | {conditions.get("initial_state",0)} | {conditions.get("execute",10)} | 미실행 | 미측정 | 미측정')
        continue
    lines.append(f"{name} | {row['initial_state_index']} | {row['execution_horizon']} | {row['status']} | {row['success']} | {row['steps']}")
    video_path = RUNS/name/'rollout.mp4'
    if video_path.exists():
        print(name)
        display(Video(url='/files/'+video_path.relative_to(ROOT).as_posix(), embed=False))
display(Markdown('\n'.join(lines)))
all_completed = not summary['missing_runs'] and all(r['status']=='completed' for r in summary['runs'])
print('계획 조건 모두 실행 완료:', all_completed)
print('집계 파일:', RUNS/'summary.json')
print('completed 실패는 실행 완료입니다. incomplete와 미실행은 완료 조건 수에 넣지 않습니다.')


## 결과 해설과 다음 단계

실행 완료와 과제 성공은 다른 열입니다. 결과가 없으면 미실행으로 남기고 성공률 분모에 섞지 않습니다.

중단 결과를 덮어쓰지 않습니다. 재시도는 새 이름으로 보존하고 평가 목록에도 추가합니다. 같은 조건 완료 결과는 재사용하며 성공할 때까지 반복하지 않습니다.

4일차의 사람이 만든 판단·제어와 5일차의 학습된 행동 예측을 입력·출력·실패 위치로 연결해 마무리합니다.

출력 표·그래프·영상과 강사 설명을 함께 확인합니다. 별도 관찰 답안을 작성하지 않습니다.
